![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 01 | Framing and Baseline

*Project 3 — AI Product: From Model to Product*

**Team:**
**Track:** model / LLM
**Data:**

---

Frame the product, look at the data, fix your evaluation before you build anything, and score the baseline your solution will have to beat.

> **Done when:** your framing is written down, you know what one row or one document is and what is wrong with the data, your test set or evaluation set is fixed, and you can say your metric and your baseline score out loud.

In [15]:
# %pip install sentence-transformers

In [16]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

---
## 0. Setup

In [17]:
df = pd.read_csv("../data/raw/jobs.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (162, 7)


,id,title,company,location,description,url,search_keyword
0,5407298685,Python Developer,Global Market Solutions Germany,"Hausen, Frankfurt am Main","GMS is an operational consultancy with expertise in data and digital systems. Whether digitalisation, data & analytics or compliance - our consultants are the missing piece of our customer’s digit...",https://www.adzuna.de/details/5407298685?utm_medium=api&utm_source=87077f8e,Python
1,5931600841,Python Entwickler (m/w/d),Instaffo GmbH,"Eschborn, Main-Taunus-Kreis",Du bist interessiert an der Stelle als Python Entwickler (m/w/d) bei just.nerds GmbH & Co. KG ? Auf Instaffo kannst du dich für diesen und weitere Jobs ohne großen Aufwand bewerben. Erstelle dein ...,https://www.adzuna.de/details/5931600841?utm_medium=api&utm_source=87077f8e,Python
2,5938275963,Consultant - Python (m/w/d),x-markets consulting,"Eschborn, Main-Taunus-Kreis","Die Boutique-Beratung für den KapitalmarktSeit nunmehr 25 Jahren gestaltet x-markets consulting als kleine, feine Beratungsboutique den Kapitalmarkt mit seinen hochwertigen Leistungen für namhafte...",https://www.adzuna.de/details/5938275963?utm_medium=api&utm_source=87077f8e,Python
3,5533884834,Senior FullStack Python & JS Developer -DEU,Cryoport Systems,"Darmstadt, Hessen","Cryoport Systems is a comprehensive supply chain partner for the life sciences industry, delivering specialized solutions to meet the challenges of the biopharmaceutical, cell and gene therapy, re...",https://www.adzuna.de/details/5533884834?utm_medium=api&utm_source=87077f8e,Python
4,5940743171,Automotive Test Automation Engineer – Python & AI,Da Vinci Engineering,"Eschborn, Main-Taunus-Kreis","Stellen-ID: 22567 Zweck und Ziel der Stelle We are searching intelligent and innovative employees. If you are interested in working for a dynamic company with flat hierarchy, we look forward to re...",https://www.adzuna.de/details/5940743171?utm_medium=api&utm_source=87077f8e,Python


In [18]:
df.isnull().sum()

id                0
title             0
company           1
location          0
description       0
url               0
search_keyword    0
dtype: int64

In [19]:
print(df["description"].str.len().describe())

print("\nExample job description:\n")
print(df["description"].iloc[0])

count    162.000000
mean     497.944444
std       22.053647
min      225.000000
25%      500.000000
50%      500.000000
75%      500.000000
max      500.000000
Name: description, dtype: float64

Example job description:

GMS is an operational consultancy with expertise in data and digital systems. Whether digitalisation, data & analytics or compliance - our consultants are the missing piece of our customer’s digital puzzle. We empower their ongoing digital transformation and transform challenges into opportunities. We're part of the MoOngy Group, a more than 18 years old European network of companies with a focus on IT & Engineering expertise. Your tasks as Python developer: Bachelor's degree in Computer Scienc…


In [20]:
print("Unique job titles:", df["title"].nunique())
print("Unique companies:", df["company"].nunique())

display(df["search_keyword"].value_counts())

Unique job titles: 131
Unique companies: 97


search_keyword
Python                           20
Data Analyst                     20
Data Scientist                   19
Tableau                          19
Software engineers               19
Cloud engineers                  19
Machine Learning                 18
Backend Developer                14
Computer programmer               7
Technical support specialists     7
Name: count, dtype: int64

In [21]:
candidate_text = """
Python developer with experience in machine learning,
data analysis, SQL, Pandas, scikit-learn, and REST APIs.
Experience developing AI applications, working with
LLMs, and deploying Python applications.
Looking for a role in data science or AI engineering.
"""

In [22]:
candidate_profiles = {
    "ml_engineer": """
        Junior machine learning engineer with Python, Pandas,
        scikit-learn, SQL, data preprocessing, model evaluation,
        and experience building predictive models.
        Looking for a junior AI or machine learning position.
    """,

    "backend_developer": """
        Backend developer with Java, Spring Boot, REST APIs,
        SQL databases, unit testing, and microservices.
        Interested in backend software engineering roles.
    """,

    "data_analyst": """
        Data analyst experienced in SQL, Excel, Tableau,
        Power BI, dashboards, statistics, and reporting.
        Looking for business intelligence or data analyst roles.
    """,

    "cloud_engineer": """
        Cloud engineer with Microsoft Azure, Docker,
        Kubernetes, CI/CD, cloud infrastructure,
        and deployment automation experience.
    """,

    "technical_support": """
        IT support specialist with experience troubleshooting
        hardware and software issues, Windows administration,
        networking, help desk, and customer support.
    """
}

In [23]:
import json
from pathlib import Path

output = Path("../data/candidate_profiles.json")

with open(output, "w", encoding="utf-8") as f:
    json.dump(candidate_profiles, f, indent=4)

print(f"Saved {len(candidate_profiles)} candidate profiles")

Saved 5 candidate profiles


In [24]:
df["job_text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("")
)

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine candidate profile and job descriptions
documents = [candidate_text] + df["job_text"].tolist()

# Convert text into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words="english")

tfidf_matrix = vectorizer.fit_transform(documents)

# Compare candidate against all jobs
similarities = cosine_similarity(
    tfidf_matrix[0:1],
    tfidf_matrix[1:]
).flatten()

df["tfidf_score"] = similarities

In [26]:
top_jobs = df.sort_values(
    "tfidf_score", ascending=False
).head(5)

display(
    top_jobs[
        ["title", "company", "location", "tfidf_score"]
    ]
)

,title,company,location,tfidf_score
17,Senior Cloud Developer Python | AI (m/w/d) 100.000€,SThree Germany,"Innenstadt, Aschaffenburg",0.163360
13,Cloud & AI Engineer Python (m/w/d) 100.000€,SThree Germany,"Sonnenberg, Wiesbaden",0.153385
14,Python Cloud Engineer - AI (m/w/d) 100.000€,SThree Germany,"Arheilgen, Darmstadt",0.153385
15,Cloud Software Engineer Python & AI (m/w/d) 100.000€,SThree Germany,"Altstadt, Mainz",0.150913
18,Cloud Software Engineer (m/w/d) Python & AI 100.000€,SThree Germany,"Niederrad, Frankfurt am Main",0.150913


In [27]:
feature_names = vectorizer.get_feature_names_out()

candidate_weights = tfidf_matrix[0].toarray().flatten()

word_importance = pd.DataFrame({
    "word": feature_names,
    "tfidf_weight": candidate_weights
})

display(
    word_importance
    .sort_values("tfidf_weight", ascending=False)
    .head(15)
)

,word,tfidf_weight
858,experience,0.408866
164,applications,0.378204
1887,python,0.227533
95,ai,0.212046
593,deploying,0.204433
618,developing,0.204433
1458,learn,0.204433
1487,llms,0.204433
1745,pandas,0.204433
1949,rest,0.204433


In [28]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pretrained embedding model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Encode candidate and job descriptions
candidate_embedding = model.encode([candidate_text])

job_embeddings = model.encode(
    df["job_text"].tolist(),
    show_progress_bar=True
)

# Calculate semantic similarity
semantic_scores = cosine_similarity(
    candidate_embedding,
    job_embeddings
).flatten()

df["semantic_score"] = semantic_scores

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [29]:
comparison = pd.DataFrame({
    "TF-IDF": df.nlargest(5, "tfidf_score")["title"].tolist(),
    "Sentence Transformers": df.nlargest(5, "semantic_score")["title"].tolist()
})

display(comparison)

,TF-IDF,Sentence Transformers
0,Senior Cloud Developer Python | AI (m/w/d) 100.000€,Python Cloud Engineer - AI (m/w/d) 100.000€
1,Cloud & AI Engineer Python (m/w/d) 100.000€,Python Developer
2,Python Cloud Engineer - AI (m/w/d) 100.000€,(Senior) Software Engineer – Java & Python (m/w/d)
3,Cloud Software Engineer Python & AI (m/w/d) 100.000€,Data Analyst - Chatbot & Voicebot (w/m/d)
4,Cloud Software Engineer (m/w/d) Python & AI 100.000€,AI Cloud Platform Engineer Python (m/w/d) 100.000€


In [30]:
comparison.to_csv(
    "../data/model_comparison_2.csv",
    index=False
)

In [31]:
# Select 30 jobs, using a fixed random seed
evaluation_jobs = df.sample(
    n=30,
    random_state=42
).copy()

# Keep columns needed for evaluation
evaluation_jobs = evaluation_jobs[
    ["id", "title", "description", "search_keyword"]
]

print("Evaluation jobs:", len(evaluation_jobs))
display(evaluation_jobs[["id", "title", "search_keyword"]])

Evaluation jobs: 30


,id,title,search_keyword
158,5923898621,Cloud Engineer (m/w/d),Cloud engineers
109,5905535959,(Junior) Consultant Financial Services - BI und Datenmodellierung (w/m/d) - KPMG,Tableau
131,5959814195,.NET Software Engineer,Software engineers
55,5802901312,Werkstudent (m/w/d) - Foresight,Machine Learning
94,5948130412,Servicetechniker - Heizung / Sanitär / Klimatechnik (m/w/d),Computer programmer
29,5937990513,Systemingenieur Mask Metrology Systeme (m/f/d),Data Scientist
101,5825712982,Head of Department (m/w/d) BI/Controlling,Tableau
51,5953445357,Senior Consultant Data Governance & Data Engineering (Collibra) (m/w/d),Machine Learning
100,5821138100,Manager (m/w/d) Data & Analytics Consulting,Tableau
144,5836626570,Cloud Engineer (all genders) in Frankfurt,Cloud engineers


In [32]:
evaluation_jobs["search_keyword"].value_counts()

search_keyword
Tableau                6
Python                 6
Cloud engineers        4
Data Scientist         4
Machine Learning       3
Software engineers     2
Computer programmer    2
Data Analyst           2
Backend Developer      1
Name: count, dtype: int64

In [33]:
display(
    evaluation_jobs[
        ["title", "search_keyword"]
    ].sort_values("search_keyword")
)

,title,search_keyword
66,Senior Developer C# Fullstack oder Backend (m/w/d),Backend Developer
158,Cloud Engineer (m/w/d),Cloud engineers
153,Senior Cloud Platform Engineer (m/w/d) CI/CD & Kubernetes,Cloud engineers
143,IT Security & Cloud Engineer (m/w/d),Cloud engineers
144,Cloud Engineer (all genders) in Frankfurt,Cloud engineers
97,Anwendungsberater (m/w/d) Software,Computer programmer
94,Servicetechniker - Heizung / Sanitär / Klimatechnik (m/w/d),Computer programmer
75,Business Process & Data Analyst bAV (m/w/d),Data Analyst
84,Kaufmännische Sachbearbeiter (m/w/d),Data Analyst
31,Elektroniker/Mechatroniker (w/m/d) für den Aufbau von komplexen Messsystemen,Data Scientist


In [34]:
role_patterns = {
    "ml_engineer": r"machine learning|data scientist|ai engineer|ki.engineer",
    "backend_developer": r"backend|java|spring|software engineer|software developer",
    "data_analyst": r"data analyst|business intelligence|bi |reporting|tableau",
    "cloud_engineer": r"cloud|devops|azure|platform engineer",
    "technical_support": r"it support|helpdesk|service desk|technical support|it administrator|systemadministrator"
}

for role, pattern in role_patterns.items():
    matches = df[
        df["title"].str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
    ]

    print(f"\n{role}: {len(matches)} matching job titles")
    display(matches[["id", "title"]].head(10))


ml_engineer: 17 matching job titles


,id,title
13,5953648768,Cloud & AI Engineer Python (m/w/d) 100.000€
20,5943389711,Data Scientist (m/w/d)
21,5953934080,Associate Data Scientist (m/w/d)
22,5062250361,Senior Data Scientist AI (m/w/d)
23,5936729063,Senior Data Scientist - Fokus Pharma (m/w/d)
25,5956672032,"(Junior) Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur"
26,5956672304,"Senior Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur"
27,5936199583,Machine Learning Engineer / Data Scientist Google Cloud (all genders)
30,5955434494,"Data Scientist für Compliance (m/w/d) - Datenbankentwicklung/BI, Risk / Compliance Management, Ingenieur"
40,5828492029,(Junior) Machine Learning Engineer (m/w/d)



backend_developer: 42 matching job titles


,id,title
7,5959431069,Backend-Entwickler (Python / PHP) (m/w/d)
8,5956294685,Backend-Entwickler (Python / PHP) (m/w/d)
9,5701642655,Backend-Entwickler (Python / PHP) (m/w/d)
10,5947098925,(Senior) Software Engineer – Java & Python (m/w/d)
11,5957955314,"Softwareentwickler Java und Python (w|m|d) - Anwendungsentwicklung, Ingenieur"
15,5953648780,Cloud Software Engineer Python & AI (m/w/d) 100.000€
18,5953648791,Cloud Software Engineer (m/w/d) Python & AI 100.000€
19,5956465101,(Senior) Data Engineer / Software Engineer – Python & Cloud (m/w/d)
44,5938631492,"Machine Learning Engineer (m/w/d) | Java, Python - 75.000€"
53,5907971202,Senior AI Software Engineer (all genders) in Frankfurt



data_analyst: 16 matching job titles


,id,title
71,5574048686,Data Analyst als Senior Consultant SAP Analytics Cloud & Power BI (m/w/d)
72,5943666507,Data Analyst - Wettbewerbsanalyse (m/w/d)
73,5950177911,Data Analyst - Marketing & Growth (w/m/d)
74,5950183749,Data Analyst - Chatbot & Voicebot (w/m/d)
75,5947325839,Business Process & Data Analyst bAV (m/w/d)
76,5945924160,Junior Data Analyst / Mitarbeiter Digitalisierung m/w/d
77,5956999493,Junior Data Analyst / Mitarbeiter Digitalisierung m/w/d
78,5751150763,Data Analyst / Power BI Professional (m/w/d)
79,5957378074,Trainee Financial Data Analyst – Wertpapierverwahrung (m/w/d)
80,5946147716,Specialist (m/w/d) AFC Performance & Reporting



cloud_engineer: 37 matching job titles


,id,title
5,5953191243,Netzwerkarchitekt - Cloud / Automation / CCNP (m/w/d)
6,5958035099,Senior Platform Engineer – Cloud & Analytics (m/w/d)
12,5889330917,DevOps Engineer (m/w/d)
13,5953648768,Cloud & AI Engineer Python (m/w/d) 100.000€
14,5953648802,Python Cloud Engineer - AI (m/w/d) 100.000€
15,5953648780,Cloud Software Engineer Python & AI (m/w/d) 100.000€
16,5953648782,AI Cloud Platform Engineer Python (m/w/d) 100.000€
17,5953648779,Senior Cloud Developer Python | AI (m/w/d) 100.000€
18,5953648791,Cloud Software Engineer (m/w/d) Python & AI 100.000€
19,5956465101,(Senior) Data Engineer / Software Engineer – Python & Cloud (m/w/d)



technical_support: 5 matching job titles


,id,title
91,5956294735,IT Support Mitarbeiter (m/w/d)
92,5959431067,IT Support Mitarbeiter (m/w/d)
117,5866031244,Aftersales Technical Support Specialist (m/w/d)
119,5959436878,"Werde IT Support Specialist (m/w/d), Quereinsteiger willkommen!"
120,5953993048,Remote Sensing Scientific and Technical Support - Passive Microwave Imaging Specialist (m/f/d) [26185]


In [35]:
selected_jobs = []

for role, pattern in role_patterns.items():

    matches = df[
        df["title"].str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
    ]

    # Select up to 6 jobs per category
    sample = matches.sample(
        n=min(6, len(matches)),
        random_state=42
    )

    selected_jobs.append(sample)

# Combine samples from all categories
evaluation_jobs = pd.concat(
    selected_jobs,
    ignore_index=True
)

# Remove jobs selected for multiple categories
evaluation_jobs = evaluation_jobs.drop_duplicates(
    subset=["id"]
).reset_index(drop=True)

print("Evaluation jobs:", len(evaluation_jobs))

display(
    evaluation_jobs[
        ["id", "title", "search_keyword"]
    ]
)

Evaluation jobs: 28


,id,title,search_keyword
0,5953648768,Cloud & AI Engineer Python (m/w/d) 100.000€,Python
1,5943389711,Data Scientist (m/w/d),Data Scientist
2,5956672032,"(Junior) Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur",Data Scientist
3,5939347586,AI Engineer (m/w/d) - Implementation & Consulting,Machine Learning
4,5892995304,Computer Vision Intern – Machine Learning Operations (m/f/d),Machine Learning
5,5727342997,Senior Machine Learning Engineer – Time Series & Forecasting (w/m/d),Machine Learning
6,5955781475,Software Engineer - Computer Vision & Künstliche Intelligenz (m/w/d),Software engineers
7,5625824028,Backend Developer (m/w/d) – IMES Industrie-Software / Microservices,Backend Developer
8,5938631492,"Machine Learning Engineer (m/w/d) | Java, Python - 75.000€",Machine Learning
9,5959814190,.NET Software Engineer,Software engineers


In [36]:
from pathlib import Path

output = Path("../data/evaluation_jobs.csv")

evaluation_jobs.to_csv(output, index=False)

print(f"Saved {len(evaluation_jobs)} jobs")

Saved 28 jobs


In [37]:
evaluation_jobs.to_csv(
    "../data/evaluation_jobs.csv",
    index=False
)

print(f"Saved {len(evaluation_jobs)} evaluation jobs")

Saved 28 evaluation jobs


In [38]:
labels = []

for profile_id in candidate_profiles:
    for _, job in evaluation_jobs.iterrows():
        labels.append({
            "profile_id": profile_id,
            "job_id": str(job["id"]),
            "job_title": job["title"],
            "relevance": ""
        })

labels_df = pd.DataFrame(labels)

labels_df.to_csv(
    "../data/relevance_labels.csv",
    index=False
)

print("Labels to assign:", len(labels_df))
display(labels_df.head())
print("Total judgments needed:", len(labels_df))

Labels to assign: 140


,profile_id,job_id,job_title,relevance
0,ml_engineer,5953648768,Cloud & AI Engineer Python (m/w/d) 100.000€,
1,ml_engineer,5943389711,Data Scientist (m/w/d),
2,ml_engineer,5956672032,"(Junior) Data Scientist (m/w/d) - Datenbankentwicklung/BI, Ingenieur",
3,ml_engineer,5939347586,AI Engineer (m/w/d) - Implementation & Consulting,
4,ml_engineer,5892995304,Computer Vision Intern – Machine Learning Operations (m/f/d),


Total judgments needed: 140


In [39]:
#%pip install ipywidgets

In [41]:
import ipywidgets as widgets

In [42]:

from IPython.display import display, clear_output

# Load our evaluation data
evaluation_jobs = pd.read_csv(
    "../data/evaluation_jobs.csv",
    dtype={"id": str}
)

labels_df = pd.read_csv(
    "../data/relevance_labels.csv",
    dtype={"job_id": str}
)

# Combine labels with job descriptions
review_df = labels_df.merge(
    evaluation_jobs[["id", "description"]],
    left_on="job_id",
    right_on="id",
    how="left"
)

# Current position in the labeling process
current_index = 0

# Interface components
output = widgets.Output()

score_selector = widgets.Dropdown(
    options=[
        ("Select relevance", None),
        ("0 - Not relevant", 0),
        ("1 - Partially relevant", 1),
        ("2 - Highly relevant", 2)
    ],
    description="Score:"
)

save_button = widgets.Button(
    description="Save & Next",
    button_style="success"
)


def show_job():
    with output:
        clear_output(wait=True)

        row = review_df.iloc[current_index]

        print(f"Job {current_index + 1} / {len(review_df)}")
        print(f"\nCandidate: {row['profile_id']}")
        print(f"\nCandidate profile:\n{candidate_profiles[row['profile_id']]}")
        print(f"\nJob: {row['job_title']}")
        print(f"\nDescription:\n{row['description']}")

        score_selector.value = (
            None if pd.isna(row["relevance"])
            else int(row["relevance"])
        )


def save_and_next(button):
    global current_index

    if score_selector.value is None:
        with output:
            print("\nPlease select a relevance score first.")
        return

    # Save the current score
    review_df.loc[current_index, "relevance"] = score_selector.value

    # Save progress after every judgment
    review_df[
        ["profile_id", "job_id", "job_title", "relevance"]
    ].to_csv(
        "../data/relevance_labels.csv",
        index=False
    )

    current_index += 1

    if current_index < len(review_df):
        show_job()
    else:
        with output:
            clear_output()
            print("All relevance labels completed!")


save_button.on_click(save_and_next)

display(output, score_selector, save_button)

show_job()

Output()

Dropdown(description='Score:', options=(('Select relevance', None), ('0 - Not relevant', 0), ('1 - Partially r…

Button(button_style='success', description='Save & Next', style=ButtonStyle())

In [43]:
labels_check = pd.read_csv("../data/relevance_labels.csv")

print("Completed:", labels_check["relevance"].notna().sum())
print("Remaining:", labels_check["relevance"].isna().sum())

Completed: 12
Remaining: 128


In [44]:
import pandas as pd

labels_df = pd.read_csv(
    "../data/relevance_labels.csv",
    dtype={"job_id": str}
)

evaluation_jobs = pd.read_csv(
    "../data/evaluation_jobs.csv",
    dtype={"id": str}
)

# Add job descriptions to make labeling easier
excel_df = labels_df.merge(
    evaluation_jobs[["id", "description"]],
    left_on="job_id",
    right_on="id",
    how="left"
)

# Add candidate profile descriptions
excel_df["candidate_profile"] = excel_df["profile_id"].map(
    candidate_profiles
)

# Keep relevant columns
excel_df = excel_df[
    [
        "profile_id",
        "candidate_profile",
        "job_id",
        "job_title",
        "description",
        "relevance"
    ]
]

excel_df.to_excel(
    "../data/relevance_labels.xlsx",
    index=False
)

print("Excel file created!")

Excel file created!


---
## 1. Frame the product

Before any code. In a few sentences:

- **The user:** who uses your product, and in what situation?
- **The input and the output:** what do they give it, and what do they get back?
- **The decision:** what do they do with the output?
- **The metric:** how will you know it works, and which mistake costs more? (*Model track:* a missed complaint or a false alarm? *LLM track:* a wrong answer, an answer with no source, or a refusal?)

You wrote a first version on launch day. Refine it here.

---
## 2. Load the data

Run `python download_data.py` (or `python download_data.py clothing`) from the repo root first. Your files land in `data/raw/`, which is gitignored.

---
## 3. First look

Shape, columns, a handful of rows **read in full**. What is one row? For the Amazon data, join a few reviews to their product in the metadata file on `parent_asin`.

---
## 4. Data quality

Empty and very short texts, HTML (`<br />`, `&#34;`), duplicates, texts in other languages, missing titles or metadata. Write down what you find and what you will do about it. Put the cleaning in a function in `src/functions.py`: the app will need it too.

---
## 5. Look at the texts

Lengths, the most frequent words, how the texts differ between ratings, products or periods. Read some hard cases: the 3-star reviews, the sarcastic ones, the ones whose rating and text disagree.

---
## 6. Model track — the target and the split

Skip this section on the LLM track.

- **Define the target** with a stated rule, for example stars to sentiment, and show the share of each class.
- **Split once**: a fixed `RANDOM_STATE`, stratified. Decide whether the split is random, by product (`parent_asin`) or by time, and say why. The test set is not touched again until notebook 03.

---
## 7. LLM track — the documents and the evaluation set

Skip this section on the model track.

- **Prepare the documents:** which texts the system will search (reviews, product descriptions, or both), how you chunk them, and the metadata each chunk keeps (product, rating, date, id), so that answers can cite them.
- **Write the evaluation set now, before you build:** at least 15 questions your user would really ask, each with the expected answer or the id of the source that answers it, and **at least 3 questions the data cannot answer**. Save it as a CSV in `data/` (it is small, and it is yours, so you can commit it).

---
## 8. The baseline

The simplest thing that could work, scored with the same evaluation as your final solution.

- **Model track:** a `DummyClassifier`, then TF-IDF with logistic regression, scored with cross-validation on the training set.
- **LLM track:** the LLM alone (no retrieval, no tools) on your evaluation set, or a keyword search (TF-IDF) for the retrieval part. Log the tokens and the cost.

---
## 9. What the baseline tells you

In prose: how good is the baseline, which mistakes does it make, and what do you expect your solution to fix?